## 面试问题

循环级发布门禁：固定任务集怎么评测一次 loop 改动的回归？

## 回答主线

改动循环后不能凭 demo 判断，要用固定任务集比较 baseline 与 candidate 的成功率/步数/成本，不回归才放行。本 Notebook 用 4 个带期望的任务集评测 baseline/candidate/bad 三个版本，门禁规则(成功率不降且平均步数不超阈值)放行 candidate、拦截成功率下降的 bad。

## 真实案例

固定任务集 4 个任务。baseline 成功 3/4；candidate 改了停止阈值成功 4/4；bad 版本成功 2/4(回归)。门禁在同一任务集上比较三者。数据为教学任务集，不代表真实评测。

In [1]:
task_set = ["t0", "t1", "t2", "t3"]  # 固定回归任务集。

results_by_version = {  # 三个版本在每个任务上的 是否成功与步数。
    "baseline": {"t0": (True, 3), "t1": (True, 4), "t2": (False, 6), "t3": (True, 3)},  # baseline 成功3/4。
    "candidate": {"t0": (True, 3), "t1": (True, 4), "t2": (True, 5), "t3": (True, 4)},  # candidate 成功4/4。
    "bad": {"t0": (True, 2), "t1": (False, 6), "t2": (False, 6), "t3": (True, 3)},  # bad 成功2/4回归。
}  # 结束版本结果定义。

print("固定任务集:", task_set)  # 展示任务集。
for v in results_by_version:  # 逐个版本。
    print("  版本", v, "结果:", results_by_version[v])  # 展示每个版本在各任务的结果。

固定任务集: ['t0', 't1', 't2', 't3']
  版本 baseline 结果: {'t0': (True, 3), 't1': (True, 4), 't2': (False, 6), 't3': (True, 3)}
  版本 candidate 结果: {'t0': (True, 3), 't1': (True, 4), 't2': (True, 5), 't3': (True, 4)}
  版本 bad 结果: {'t0': (True, 2), 't1': (False, 6), 't2': (False, 6), 't3': (True, 3)}


## 基线（Baseline）

反面基线：只用单个任务的 demo 判断。candidate 在 t0 上成功，看着没问题，但完全覆盖不到 t1/t2/t3 上的回归风险。

In [2]:
def single_demo(version_results, task):  # 只用单个任务判断版本好坏。
    ok, steps = version_results[task]  # 取该任务结果。
    return {"task": task, "success": ok, "steps": steps}  # 返回单任务结论。

demo = single_demo(results_by_version["candidate"], "t0")  # 只看 candidate 在 t0 的表现。
print("单 demo(candidate, t0):", demo)  # 展示单个例子看着成功。
print("但单例覆盖不了 t1/t2/t3 的回归风险")  # 说明缺陷。

单 demo(candidate, t0): {'task': 't0', 'success': True, 'steps': 3}
但单例覆盖不了 t1/t2/t3 的回归风险


## 失败案例与修正

单 demo 漏掉回归。修正是固定任务集 + 门禁：在同一任务集上评测成功率与平均步数，门禁要求成功率不降且步数不超阈值，能放行 candidate、拦截 bad。

In [3]:
def evaluate(version_results, tasks):  # 在固定任务集上评测一个版本。
    successes = 0  # 成功任务数。
    total_steps = 0  # 总步数。
    for t in tasks:  # 遍历每个任务。
        ok, steps = version_results[t]  # 取该任务结果。
        if ok:  # 成功计数。
            successes += 1  # 累加成功。
        total_steps += steps  # 累加步数。
    success_rate = successes / len(tasks)  # 成功率。
    avg_steps = total_steps / len(tasks)  # 平均步数。
    return {"success_rate": success_rate, "avg_steps": avg_steps}  # 返回指标。

def gate(baseline_metrics, candidate_metrics, max_avg_steps=5.0):  # 发布门禁规则。
    no_regression = candidate_metrics["success_rate"] >= baseline_metrics["success_rate"]  # 成功率不下降。
    within_cost = candidate_metrics["avg_steps"] <= max_avg_steps  # 平均步数不超阈值。
    passed = no_regression and within_cost  # 同时满足才放行。
    return {"passed": passed, "no_regression": no_regression, "within_cost": within_cost}  # 返回门禁结论。

In [4]:
base_m = evaluate(results_by_version["baseline"], task_set)  # 评测 baseline。
cand_m = evaluate(results_by_version["candidate"], task_set)  # 评测 candidate。
bad_m = evaluate(results_by_version["bad"], task_set)  # 评测 bad 版本。
cand_gate = gate(base_m, cand_m)  # candidate 过门禁。
bad_gate = gate(base_m, bad_m)  # bad 过门禁。
print("baseline 指标:", base_m)  # 展示 baseline 指标。
print("candidate 指标:", cand_m, "门禁通过:", cand_gate["passed"])  # 展示 candidate 通过。
print("bad 指标:", bad_m, "门禁通过:", bad_gate["passed"])  # 展示 bad 被拦截。

baseline 指标: {'success_rate': 0.75, 'avg_steps': 4.0}
candidate 指标: {'success_rate': 1.0, 'avg_steps': 4.0} 门禁通过: True
bad 指标: {'success_rate': 0.5, 'avg_steps': 4.25} 门禁通过: False


In [5]:
print("单 demo 覆盖任务数: 1 / 固定任务集:", len(task_set))  # 单 demo 覆盖不足。
print("candidate 成功率", cand_m["success_rate"], ">= baseline", base_m["success_rate"], "-> 放行:", cand_gate["passed"])  # candidate 放行。
print("bad 成功率", bad_m["success_rate"], "< baseline", base_m["success_rate"], "-> 拦截:", not bad_gate["passed"])  # bad 拦截。

单 demo 覆盖任务数: 1 / 固定任务集: 4
candidate 成功率 1.0 >= baseline 0.75 -> 放行: True
bad 成功率 0.5 < baseline 0.75 -> 拦截: True


## 结果解读

单 demo 看 candidate 在 t0 成功、漏掉全局；固定任务集显示 candidate 成功率 1.0≥baseline 0.75、平均步数 4.0≤阈值 5.0，放行；bad 成功率 0.5<0.75，门禁 no_regression=False 拦截。要点：同集对比、多指标、门禁能拦回归、任务集防过拟合。这也是整个专题的收口——用重放/trace/终止/成本机制评测一次改动。

In [6]:
assert base_m["success_rate"] == 0.75  # baseline 成功率 3/4。
assert cand_m["success_rate"] == 1.0  # candidate 成功率 4/4。
assert cand_gate["passed"] is True  # candidate 不回归且在成本内应放行。
assert bad_gate["passed"] is False  # bad 成功率下降应被拦截。
assert bad_gate["no_regression"] is False  # bad 的回归被门禁识别。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
